# Reichman University NLP Project

Cross-request **token reuse** across LLM workloads. Every number in the final
table is computed by the code in this notebook — each cell loads a corpus,
applies its treatment, and measures reuse with the shared accounting core in
`core/` (`measure_disjoint`: second-occurrence reuse, whole-stream denominator,
one trajectory per task, ≥500-token spans, prefix-cache-disjoint credit).

A corpus whose data isn't present is skipped (not faked). HuggingFace-hosted
corpora download on first run — set `HF_TOKEN`. Run the cells top to bottom.


## Setup and shared measurement core

In [ ]:
import os, sys, re, json, glob
from collections import Counter, defaultdict

ROOT = os.getcwd()                          # run this notebook from the repo root
sys.path.insert(0, os.path.join(ROOT, "core"))
sys.path.insert(0, os.path.join(ROOT, "experiments"))
import pandas as pd

from data import load_hf_token; load_hf_token()
from pic_rules import lines_of, nt, MIN                 # tokenizer + >=500 span units
from prefix_disjoint import measure_disjoint            # prefix-vs-PIC disjoint accounting
from analyze_rules import run as rules_run              # whole-prompt five-rule pass (coding)
from analyze_mined import (apply_spec, OBS,
    sessions_nnetnav, sessions_mind2web,
    sessions_itbench_ciso, sessions_itbench_alerts)
from rag500 import sessions_multidoc2dial, sessions_mtrag, tau_docs
from huggingface_hub import HfApi, hf_hub_download

ROWS, SKIPPED = [], []

def add(family, corpus, treat, prefix, pic, pic_proc, in_sess, cross_sess):
    ROWS.append({"Workload family": family, "Corpus": corpus,
                 "Prefix caching": prefix, "PIC": pic, "PIC proc.": pic_proc,
                 "in-sess.": in_sess, "cross-sess.": cross_sess, "Treatment": treat})

def stage(name):
    # Run a corpus's producing code now; record it, or note a skip if its data is absent.
    def deco(fn):
        try:
            fn(); print("OK  ", name)
        except Exception as e:
            SKIPPED.append((name, repr(e))); print("skip", name, "->", type(e).__name__)
    return deco

## Chat prompts — WildChat-1M, PAWS

In [ ]:
from data import iter_first_turns

@stage("WildChat-1M (40K)")
def _():
    texts = [r["text"] for r in iter_first_turns(max_rows=40_000)]
    d = measure_disjoint([{"task": i, "snaps": [lines_of(t)]} for i, t in enumerate(texts)])
    add("Chat prompts", "WildChat-1M (40K)", "none",
        d["prefix_caching_pct"], d["pic_pct"], d["pic_pct"], d["in_session"], d["cross_session"])

@stage("PAWS (40K)")
def _():
    from datasets import load_dataset
    ds = load_dataset("google-research-datasets/paws", "labeled_final",
                      split="train", streaming=True)
    texts, n = [], 0
    for r in ds:
        for k in ("sentence1", "sentence2"):
            if r.get(k):
                texts.append(r[k]); n += 1
        if n >= 40_000:
            break
    d = measure_disjoint([{"task": i, "snaps": [lines_of(t)]} for i, t in enumerate(texts)])
    add("Paraphrase traffic", "PAWS (40K)", "none",
        d["prefix_caching_pct"], d["pic_pct"], d["pic_pct"], d["in_session"], d["cross_session"])

## Coding agents — SWE-smith, OpenHands, CC-Bench, SWE-agent

Whole-prompt reuse under the five rules (`analyze_rules.run`): the system prompt
is credited to prefix caching, PIC spans are cut from each newly appended message.

In [ ]:
for corpus, label in [("swesmith", "SWE-smith (1K)"), ("openhands", "OpenHands (1K)"),
                      ("ccbench", "CC-Bench (74)"), ("sweagent", "SWE-agent (1K)")]:
    @stage(label)
    def _(corpus=corpus, label=label):
        r = rules_run(corpus, "Qwen/Qwen3-0.6B", max_trajs=1000,
                      one_per_task=True, system_as_prefix=True)
        add("Coding agents", label, "none", r["system_prefix_pct"], r["reuse_pct"],
            r["reuse_pct"], r["in_session_pct"], r["cross_session_pct"])

## Web agents — NNetNav-WA, NNetNav-Live, Mind2Web

Treatment `replace`: rewrite volatile element handles / `backend_node_id`s into
stable content-derived handles so identical widgets hash alike across steps.
WebArena and NNetNav-Live share a fixed instruction head per step, so they use
the head-counted-once denominator (`web_sessionhead.run`); Mind2Web (raw HTML,
no repeated head) uses the per-snapshot disjoint measure.

In [ ]:
from web_sessionhead import run as web_run, replace_handles

for repo, family, corpus in [("stanfordnlp/nnetnav-wa", "Web pages (a11y)", "NNetNav-WA (WebArena)"),
                             ("stanfordnlp/nnetnav-live", "Web pages (live)", "NNetNav-Live")]:
    @stage(corpus)
    def _(repo=repo, family=family, corpus=corpus):
        v = web_run(repo, None)                 # untreated -> PIC
        t = web_run(repo, replace_handles)      # replace   -> PIC proc, prefix, in/cross
        add(family, corpus, "replace", t["prefix_caching_pct"], v["pic_pct"],
            t["pic_pct"], t["pic_in_sess_pct"], t["pic_cross_sess_pct"])

IDATTR = re.compile(r'\s*backend_node_id="[^"]*"')

def h64(s):
    h = 1469598103934665603
    for c in s.encode():
        h = ((h ^ c) * 1099511628211) & (1 << 64) - 1
    return h

def replace_ids(html):                      # Mind2Web: backend_node_id -> content handle
    occ = Counter()
    def sub(m):
        seg = m.group(0); cleaned = IDATTR.sub("", seg); occ[cleaned] += 1
        return re.sub(r'backend_node_id="[^"]*"',
                      f'backend_node_id="a{h64(cleaned + "|" + str(occ[cleaned]-1)):016x}"', seg)
    return re.sub(r"<[^>]*backend_node_id=\"[^\"]*\"[^>]*>", sub, html)

@stage("Mind2Web")
def _():
    m2 = sessions_mind2web(25)
    def m(xform):
        return measure_disjoint([{"task": i, "snaps": [lines_of(xform(sn) if xform else sn)
                                                       for sn in snaps]}
                                 for i, snaps in enumerate(m2)])
    v, t = m(None), m(replace_ids)
    add("Web pages (HTML)", "Mind2Web", "replace",
        v["prefix_caching_pct"], v["pic_pct"], t["pic_pct"], t["in_session"], t["cross_session"])

## Operational telemetry — ITBench SRE / K8s / alerts, BGL syslog

`relocate` moves volatile fields into a sidecar (still in the denominator) so the
stable body forms reusable spans; SRE and BGL use it, K8s/alerts don't.

In [ ]:
@stage("ITBench SRE")
def _():
    TASK = re.compile(r"(Scenario-\d+)"); REPO = "ibm-research/ITBench-Trajectories"
    files = sorted(s.rfilename for s in HfApi().dataset_info(REPO).siblings
                   if s.rfilename.endswith("session.jsonl"))
    sess = []
    for rf in files:
        p = hf_hub_download(REPO, rf, repo_type="dataset")
        m = TASK.search(rf); task = m.group(1) if m else rf
        raw_s, rel_s, side = [], [], 0
        for line in open(p):
            try: r = json.loads(line)
            except ValueError: continue
            pl = r.get("payload") or {}
            if pl.get("type") != "function_call_output": continue
            o = pl.get("output") or ""; raw = o
            try:
                inner = json.loads(o)
                if isinstance(inner, dict) and isinstance(inner.get("output"), str) \
                        and inner.get("metadata") is not None:
                    o = inner["output"]; side += nt(json.dumps(inner["metadata"], ensure_ascii=False))
                else:
                    o = json.dumps(inner, ensure_ascii=False); raw = o
            except (ValueError, TypeError): pass
            if len(o) >= 80:
                rel_s.append(lines_of(o)); raw_s.append(lines_of(raw))
        if len(rel_s) >= 2:
            sess.append({"task": task, "raw": raw_s, "rel": rel_s, "side": side})
        if len(sess) >= 40: break
    seen = set(); dd = [s for s in sess if not (s["task"] in seen or seen.add(s["task"]))]
    v = measure_disjoint([{"task": s["task"], "snaps": s["raw"]} for s in dd])
    t = measure_disjoint([{"task": s["task"], "snaps": s["rel"]} for s in dd],
                         sidecar=sum(s["side"] for s in dd))
    add("Op. telemetry", "ITBench SRE", "relocate",
        v["prefix_caching_pct"], v["pic_pct"], t["pic_pct"], t["in_session"], t["cross_session"])

@stage("ITBench infra compliance (K8s)")
def _():
    s = sessions_itbench_ciso(200)
    d = measure_disjoint([{"task": i, "snaps": [lines_of(sn) for sn in snaps]}
                          for i, snaps in enumerate(s)])
    add("Op. telemetry", "ITBench infra compliance (K8s)", "none",
        d["prefix_caching_pct"], d["pic_pct"], d["pic_pct"], d["in_session"], d["cross_session"])

@stage("ITBench alerts")
def _():
    s = sessions_itbench_alerts(40)
    d = measure_disjoint([{"task": i, "snaps": [lines_of(sn) for sn in snaps]}
                          for i, snaps in enumerate(s)])
    add("Op. telemetry", "ITBench alerts", "none",
        d["prefix_caching_pct"], d["pic_pct"], d["pic_pct"], d["in_session"], d["cross_session"])

@stage("BGL syslog (LogHub)")
def _():
    POS = (1, 4)
    lines = [l.rstrip("\n") for l in open("data/loghub/BGL_200k.log") if l.strip()]
    per = len(lines) // 20
    raw, rel, side = [], [], 0
    for s in range(20):
        chunk = lines[s*per:(s+1)*per]; rs, ts = [], []
        for i in range(0, len(chunk), 500):
            ru, tu = [], []
            for line in chunk[i:i+500]:
                toks = line.split()
                ru.append(line + "\n")
                if len(toks) >= 8:
                    side += nt(" ".join(toks[j] for j in POS if j < len(toks)))
                    tu.append(" ".join(t for j, t in enumerate(toks) if j not in POS) + "\n")
                else:
                    tu.append(line + "\n")
            rs.append(ru); ts.append(tu)
        raw.append({"task": s, "snaps": rs}); rel.append({"task": s, "snaps": ts})
    v = measure_disjoint(raw)
    t = measure_disjoint(rel, sidecar=side)
    add("Op. telemetry", "BGL syslog (LogHub)", "relocate",
        v["prefix_caching_pct"], v["pic_pct"], t["pic_pct"], t["in_session"], t["cross_session"])

## Stateful API responses — AppWorld, tau2 (airline / retail / telecom)

AppWorld uses `mask` (blank a volatile JSON field, diagnostic); tau2 is untreated.

In [ ]:
@stage("AppWorld")
def _():
    MSG = re.compile(r"llm\.input_messages\.(\d+)\.message\.role")
    best = {}
    for line in open("data/appworld/hf_traces/halo_gemini3flash_traces.jsonl"):
        r = json.loads(line); a = r.get("attributes") or {}
        if a.get("openinference.span.kind") != "LLM": continue
        idxs = [int(m.group(1)) for k in a for m in [MSG.fullmatch(k)] if m]
        if idxs:
            tid = r["trace_id"]
            if tid not in best or len(idxs) > best[tid][0]:
                best[tid] = (len(idxs), a, max(idxs))
    aw = []
    for tid, (n, a, mx) in best.items():
        task, snaps = None, []
        for i in range(mx + 1):
            role = a.get(f"llm.input_messages.{i}.message.role")
            c = a.get(f"llm.input_messages.{i}.message.content") or ""
            if role == "user":
                m = re.search(r"# Real Task Instruction\n(.*?)(?:\n\n|$)", c, re.S)
                if m: task = m.group(1).strip()
            elif role == "tool" and isinstance(c, str) and len(c) >= 80:
                snaps.append(c)
        if len(snaps) >= 2 and task:
            aw.append({"task": task, "snaps": snaps})
    seen = set(); aw = [x for x in aw if not (x["task"] in seen or seen.add(x["task"]))]
    def m(spec):
        return measure_disjoint([{"task": x["task"],
            "snaps": [lines_of(apply_spec(sn, spec) if spec else sn) for sn in x["snaps"]]}
            for x in aw])
    v, t = m(None), m(["json:release_date"])
    add("Stateful API", "AppWorld", "mask",
        v["prefix_caching_pct"], v["pic_pct"], t["pic_pct"], t["in_session"], t["cross_session"])

STD = re.compile(r"_(airline|retail|telecom)_")
for domain, pub in [("airline", "tau2 airline"), ("retail", "tau2 retail"),
                    ("telecom", "tau2 telecom")]:
    @stage(pub)
    def _(domain=domain):
        fs = [f for f in sorted(glob.glob("data/tau2/final/*.json"))
              if (m := STD.search(os.path.basename(f))) and m.group(1) == domain]
        sess = []
        for f in fs:
            for s in (json.load(open(f)).get("simulations") or []):
                snaps = [c for msg in (s.get("messages") or [])
                         if msg.get("role") == "tool" and not msg.get("error")
                         and isinstance(c := msg.get("content"), str) and len(c) >= 80]
                if len(snaps) >= 2:
                    sess.append({"task": s.get("task_id"), "snaps": snaps})
        sess.sort(key=lambda s: str(s["task"]))
        seen = set(); dd = [s for s in sess if not (s["task"] in seen or seen.add(s["task"]))]
        d = measure_disjoint([{"task": s["task"], "snaps": [lines_of(x) for x in s["snaps"]]}
                              for s in dd])
        add("Stateful API", f"tau2 {domain}", "none",
            d["prefix_caching_pct"], d["pic_pct"], d["pic_pct"], d["in_session"], d["cross_session"])

## Retrieved documents — MT-RAG, MultiDoc2Dial, tau-Knowledge

tau-Knowledge uses `align`: pack the KB into fixed ≥500-token buckets so a
document reused across tasks always lands on the same span grid.

In [ ]:
@stage("MT-RAG ibmcloud")
def _():
    bycol = defaultdict(list)
    for s in sessions_mtrag("human"):
        bycol[s["task"].split("/")[0]].append(s)
    col = next(c for c in bycol if "ibmcloud" in c)
    d = measure_disjoint([{"task": s["task"], "snaps": s["snaps"]} for s in bycol[col]])
    add("Retrieved docs", "MT-RAG ibmcloud", "none",
        d["prefix_caching_pct"], d["pic_pct"], d["pic_pct"], d["in_session"], d["cross_session"])

@stage("MultiDoc2Dial")
def _():
    m2 = sessions_multidoc2dial()
    seen = set(); m2 = [x for x in m2 if not (x["task"] in seen or seen.add(x["task"]))]
    d = measure_disjoint(m2)
    add("Retrieved docs", "MultiDoc2Dial", "none",
        d["prefix_caching_pct"], d["pic_pct"], d["pic_pct"], d["in_session"], d["cross_session"])

@stage("tau-Knowledge")
def _():
    docs = tau_docs()
    tasks = json.load(open("data/tauknowledge/banking_knowledge/tasks.json"))
    if isinstance(tasks, dict):
        tasks = tasks.get("tasks") or list(tasks.values())
    sel = [{"task": t.get("id"), "req": r} for t in tasks
           if (r := [d for d in sorted(t.get("required_documents") or []) if d in docs])]
    concat = measure_disjoint(
        [{"task": t["task"], "snaps": [[u for d in t["req"] for u in lines_of(docs[d])]]}
         for t in sel])
    buckets, buf, n = [], [], 0
    for d in sorted(docs):
        buf.append(d); n += nt(docs[d])
        if n >= MIN:
            buckets.append(buf); buf = []; n = 0
    if buf:
        buckets[-1].extend(buf) if buckets else buckets.append(buf)
    owner = {d: i for i, b in enumerate(buckets) for d in b}
    aligned = measure_disjoint(
        [{"task": t["task"], "snaps": [[u for d in buckets[i] for u in lines_of(docs[d])]
                                       for i in sorted({owner[d] for d in t["req"]})]}
         for t in sel])
    add("Retrieved docs", "tau-Knowledge", "align",
        concat["prefix_caching_pct"], concat["pic_pct"], aligned["pic_pct"],
        aligned["in_session"], aligned["cross_session"])

## Database schemas — Spider 2.0, BIRD, LiveSQLBench, BIRD-INTERACT

Schema DDL packed into ≥500-token spans reused across questions on the same
database. Prefix caching is reported schema-first / schema-after-context. These
reuse the corpus loaders in `experiments/` (schema extraction) and their
prefix/PIC measures.

In [ ]:
import spider2, bird, bird_interact
import livesql_common as lc

@stage("Spider 2.0")
def _():
    pr = spider2.load_prompts()
    pic = spider2.pic_measure(pr)
    add("Database schemas", "Spider 2.0", "none",
        f"{spider2.prefix_measure(pr, True)}/{spider2.prefix_measure(pr, False)}",
        pic["total"], pic["total"], pic["same_session"], pic["cross_sessions"])

@stage("BIRD")
def _():
    pr = bird.load_prompts()
    pic = bird.pic_measure(pr)
    add("Database schemas", "BIRD", "none",
        f"{bird.prefix_measure(pr, True)}/{bird.prefix_measure(pr, False)}",
        pic["total"], pic["total"], pic["same_session"], pic["cross_task"])

@stage("LiveSQLBench")
def _():
    root = "data/livesqlbench/large-v1"
    tasks = [json.loads(l) for l in open(os.path.join(root, "livesqlbench_large_v1_data.jsonl"))]
    cache, prompts = {}, []
    for t in tasks:
        db = t["selected_database"]
        if db not in cache:
            u = lc.db_context_units(root, db); cache[db] = (u, "\n".join(u))
        u, ctx = cache[db]
        prompts.append({"task": t["instance_id"], "units": u, "ctx": ctx,
                        "head": "Task " + t["instance_id"] + "\nQuestion: " + t["query"]})
    pic = lc.pic_measure(prompts)
    add("Database schemas", "LiveSQLBench", "none",
        f"{lc.prefix_measure(prompts, True)}/{lc.prefix_measure(prompts, False)}",
        pic["total"], pic["total"], pic["same_task"], pic["cross_task"])

@stage("BIRD-INTERACT")
def _():
    prompts = bird_interact.load("data/birdinteract/lite")
    if not prompts:
        raise FileNotFoundError("data/birdinteract/lite")
    pic = lc.pic_measure(prompts)
    add("Conversational SQL", "BIRD-INTERACT", "none",
        f"{lc.prefix_measure(prompts, True)}/{lc.prefix_measure(prompts, False)}",
        pic["total"], pic["total"], pic["same_task"], pic["cross_task"])

## Results table

In [ ]:
PAPER = {  # published PIC-processed %, for a reproduction check only
    "WildChat-1M (40K)": 3.28, "PAWS (40K)": 0.00, "SWE-smith (1K)": 17.06,
    "OpenHands (1K)": 2.69, "CC-Bench (74)": 16.43, "SWE-agent (1K)": 19.27,
    "NNetNav-WA (WebArena)": 10.03, "Mind2Web": 42.94, "NNetNav-Live": 14.79,
    "ITBench SRE": 7.71, "ITBench infra compliance (K8s)": 26.73, "ITBench alerts": 16.93,
    "BGL syslog (LogHub)": 8.49, "AppWorld": 0.40, "tau2 airline": 1.31,
    "tau2 retail": 23.08, "tau2 telecom": 3.73, "MT-RAG ibmcloud": 13.76,
    "MultiDoc2Dial": 39.97, "tau-Knowledge": 47.74, "Spider 2.0": 86.95,
    "BIRD": 78.54, "LiveSQLBench": 95.86, "BIRD-INTERACT": 97.87,
}

df = pd.DataFrame(ROWS)
if not df.empty:
    df["Paper PIC proc."] = df["Corpus"].map(PAPER)
    df["Δ"] = (df["PIC proc."] - df["Paper PIC proc."]).round(2)

def bold_family_best(col):
    best = df.groupby("Workload family")["PIC proc."].transform("max")
    return ["font-weight: bold" if v == b else "" for v, b in zip(col, best)]

print(f"produced {len(ROWS)} / 24 rows; skipped {len(SKIPPED)} (missing data)")
df.style.apply(bold_family_best, subset=["PIC proc."]).format(precision=2).hide(axis="index")

Every value above was produced by the cells in this notebook. `PIC proc.` is
reuse after the corpus's treatment; `Δ` is its gap to the published figure (0.00
means an exact reproduction). PIC savings are **in addition** to prefix caching.
Rows in `SKIPPED` had no local data available in this environment.